# 04 — 条件Cの評価と条件Dの学習・比較

前半Notebookが生成した `official6 manifest + Large feature cache + parity.json` だけを入力にします。
音声取得、音声参照、manifest生成、特徴抽出はこのNotebookでは行いません。

Cは公式9クラスheadを固定し、Dは同じheadの対象6行だけをHCUDBで更新します。真値は
angry/disgusted/fearful/happy/sad/surprisedの6クラス、損失と予測は公式9 logits全体を使います。
neutral/other/unknownが最大ならそのまま誤分類として保存します。嫌い→disgustedは研究上の近似です。
固定3行のparameter保持は、その予測率や英語性能の保持を意味しません。

独立したWSL環境に `requirements-official.txt` を導入し、`emotion2vec-official` kernelを選択してください。
全実行フラグは初期値Falseです。


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if not (ROOT / 'ser_pipeline').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'ser_pipeline').is_dir():
    raise RuntimeError('リポジトリまたはnotebooksディレクトリから実行してください。')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ser_pipeline.cache import ShardedFeatureStore
from ser_pipeline.contracts import label_profile_for_mapping_version
from ser_pipeline.diagnostics import OfficialTrainingDiagnosticsConfig
from ser_pipeline.manifest import load_manifest, validate_manifest
from ser_pipeline.official import load_official_head, require_parity_report
from ser_pipeline.study import (
    DatasetArtifacts,
    run_official_c_evaluations,
    run_official_d_evaluations,
    run_official_study,
)
from ser_pipeline.training import TrainingConfig, train_official_decoder

REVISION = '6c303ba987b86b93193de93e34bb2b077a6bedc4'
HF_HOME = Path('/mnt/c/Users/RD004/.cache/huggingface/hub') if sys.platform != 'win32' else Path.home() / '.cache/huggingface/hub'
SNAPSHOT = Path(os.environ.get('SER_OFFICIAL_SNAPSHOT', str(HF_HOME / 'models--emotion2vec--emotion2vec_plus_large' / 'snapshots' / REVISION)))
OUTPUT = ROOT / 'runs' / 'official_cd'
PARITY_REPORT = OUTPUT / 'parity.json'
MANIFEST_DIR = ROOT / 'runs' / 'ser_manifests'
MANIFESTS = {
    'msp_podcast': MANIFEST_DIR / 'msp_podcast_official6_v1.jsonl',
    'hcudb1': MANIFEST_DIR / 'hcudb1_official6_v1.jsonl',
}
CACHES = {dataset: OUTPUT / 'cache' / dataset for dataset in MANIFESTS}
OFFICIAL_ARTIFACT_CONTRACT = {
    'manifests': MANIFESTS,
    'caches': CACHES,
    'parity_report': PARITY_REPORT,
}

DEVICE = 'cpu'
SEEDS = (42, 43, 44)
CONFIG = TrainingConfig(epochs=10, batch_size=8, learning_rate=0.001, weight_decay=0, device=DEVICE)
DIAGNOSTICS_CONFIG = OfficialTrainingDiagnosticsConfig(
    tail_epochs=3,
    min_score_delta=0.02,
    min_loss_delta=0.03,
    low_train_score_threshold=0.50,
)
STUDY_OUTPUT = OUTPUT / 'study'
STUDY_SUMMARY = STUDY_OUTPUT / 'official_study_summary.json'
C_EVALUATION_OUTPUT = OUTPUT / 'evaluation_c'
C_SUMMARY = C_EVALUATION_OUTPUT / 'c_evaluation_summary.json'
D_EVALUATION_OUTPUT = OUTPUT / 'evaluation_d'
D_SUMMARY = D_EVALUATION_OUTPUT / 'd_evaluation_summary.json'
RESUME_CHECKPOINT = None
RESUME_OUTPUT = OUTPUT / 'resumed'
RESUME_SEED = 42

RUN_INPUT_ARTIFACT_CHECK = False
RUN_C_EVALUATION = False
RUN_D_TRAINING = False
RUN_D_RESUME = False
RUN_D_EVALUATION = False

def require_official_inputs():
    missing = []
    for path in [*MANIFESTS.values(), *(root / 'cache_meta.json' for root in CACHES.values()), PARITY_REPORT]:
        if not path.is_file():
            missing.append(str(path))
    if missing:
        raise FileNotFoundError(
            '前半Notebookの受け渡しartifactが不足しています（official6 manifest + Large feature cache + parity.json）: '
            + ', '.join(missing)
        )

    official_head = load_official_head(SNAPSHOT)
    resolved = {}
    report = {'contract': 'official6 manifest + Large feature cache + parity.json', 'datasets': {}}
    for dataset, manifest_path in MANIFESTS.items():
        validate_manifest(manifest_path)
        rows = load_manifest(manifest_path)
        profiles = {label_profile_for_mapping_version(row['mapping_version']) for row in rows}
        if profiles != {'official6'}:
            raise ValueError(f'{dataset} manifestはofficial6ラベル契約ではありません。')
        store = ShardedFeatureStore(CACHES[dataset], manifest_path)
        require_parity_report(official_head, PARITY_REPORT, store.meta)
        resolved[dataset] = DatasetArtifacts(manifest_path, CACHES[dataset])
        report['datasets'][dataset] = store.validation_report
    return resolved, official_head, report


## 1. 入力artifactの確認

`RUN_INPUT_ARTIFACT_CHECK` は、前半Notebookとの受け渡し契約を単独で確認するためのフラグです。
C評価、Dの学習・再開・評価も同じpreflightを必ず先に実行するため、未確認の入力で計算を開始しません。


In [ ]:
if RUN_INPUT_ARTIFACT_CHECK:
    _, _, input_report = require_official_inputs()
    print(json.dumps(input_report, ensure_ascii=False, indent=2))
else:
    print('入力artifact確認は未実行です。')


## 2. 条件Cの評価

公式headを固定したCを、MSP-Podcast Test1とHCUDB Testで各1回だけ評価します。
Dのstudy summaryやcheckpointは読みません。結果はDと分離したディレクトリへ保存します。


In [ ]:
if RUN_C_EVALUATION:
    resolved_artifacts, official_head, _ = require_official_inputs()
    c_summary = run_official_c_evaluations(
        resolved_artifacts, C_EVALUATION_OUTPUT, official_head, PARITY_REPORT,
        batch_size=CONFIG.batch_size, device=DEVICE,
    )
    for evaluation in c_summary['evaluations']:
        result = evaluation['result']
        print(f"C {result['dataset']} / {result['split']}")
        print(json.dumps(result['metrics_target6'], ensure_ascii=False, indent=2))
    print('C summary:', c_summary['summary_path'])
else:
    print('条件Cの評価は未実行です。Dのcheckpointは不要です。')


## 3. Cの結果確認と基準決定

保存済みCのUAR、macro F1、accuracy、loss、クラス別結果を確認し、Dとの比較基準を決めます。
確認済みフラグや数値による自動合否判定は設けません。このセルと次のD学習セルを分けて運用します。


In [ ]:
if C_SUMMARY.is_file():
    saved_c = json.loads(C_SUMMARY.read_text(encoding='utf-8'))
    if saved_c.get('status') != 'complete':
        raise ValueError('C summaryが完了状態ではありません。')
    for evaluation in saved_c['evaluations']:
        result = evaluation['result']
        print(f"確認対象 C {result['dataset']} / {result['split']}")
        print(json.dumps(result['metrics_target6'], ensure_ascii=False, indent=2))
else:
    print('C summaryはまだありません。先にRUN_C_EVALUATION=TrueでC評価セルだけを実行してください。')


## 4. 条件Dの学習と再開

3 seedのbest選択にはHCUDB validationだけを使います。再開時も同じ入力契約と診断設定を適用します。


In [ ]:
if RUN_D_TRAINING:
    resolved_artifacts, official_head, _ = require_official_inputs()
    summary = run_official_study(resolved_artifacts['hcudb1'], STUDY_OUTPUT, official_head, PARITY_REPORT,
                                 seeds=SEEDS, config=CONFIG, diagnostics_config=DIAGNOSTICS_CONFIG)
    print('Dの学習完了。test評価は未実行:', summary['summary_path'])
    print('診断集計（判定は助言専用）:')
    print(json.dumps(summary['diagnostics_aggregate'], ensure_ascii=False, indent=2))
    for run in summary['runs']:
        print(f"seed={run['seed']} 診断:", run['diagnostics']['path'])
        print(json.dumps(run['diagnostics']['judgement'], ensure_ascii=False, indent=2))
else:
    print('Dの3 seed学習は未実行です。best選択にはHCUDB validationだけを使います。')


In [ ]:
if RUN_D_RESUME:
    from dataclasses import replace
    if RESUME_CHECKPOINT is None:
        raise ValueError('RESUME_CHECKPOINTを設定してください。')
    resolved_artifacts, official_head, _ = require_official_inputs()
    artifact = resolved_artifacts['hcudb1']
    resumed = train_official_decoder(artifact.manifest_path, artifact.cache_root, RESUME_OUTPUT,
        official_head, PARITY_REPORT, config=replace(CONFIG, seed=RESUME_SEED),
        resume_checkpoint=RESUME_CHECKPOINT, diagnostics_config=DIAGNOSTICS_CONFIG)
    # train_official_decoderはresume後の全履歴から同じrunの診断artifactを再生成する。
    print('resume後に再生成した診断:', resumed['diagnostics']['artifacts']['training_diagnostics_json'])
    print(json.dumps(resumed['diagnostics']['judgement'], ensure_ascii=False, indent=2))
    # 再開結果のbestをD評価に使う場合は、study summaryの該当seedとhashを明示的に確定する。


## 5. 条件Dの評価と保存済みCとの比較

全seedのD bestを確定してから、MSP Test1とHCUDB TestについてDだけをseed別に評価します。
Cは再推論せず、保存済みC summaryのhead・cache・test集合の署名を現在の入力と照合して比較します。


In [ ]:
if RUN_D_EVALUATION:
    resolved_artifacts, official_head, _ = require_official_inputs()
    saved = json.loads(STUDY_SUMMARY.read_text())
    if len(saved['runs']) != len(SEEDS) or {r['seed'] for r in saved['runs']} != set(SEEDS):
        raise ValueError('全seedのD bestが揃ってからD評価してください。')
    d_summary = run_official_d_evaluations(
        resolved_artifacts, {run['seed']: run['best'] for run in saved['runs']},
        D_EVALUATION_OUTPUT, official_head, PARITY_REPORT, C_SUMMARY,
        batch_size=CONFIG.batch_size, device=DEVICE,
    )
    for evaluation in d_summary['evaluations']:
        result = evaluation['result']
        print(f"D {result['dataset']} / seed={result['seed']}")
        print(json.dumps(result['metrics_target6'], ensure_ascii=False, indent=2))
    print('C/D比較:')
    print(json.dumps(d_summary['comparisons'], ensure_ascii=False, indent=2))
    print('D summary:', d_summary['summary_path'])
else:
    print('条件Dの評価は未実行です。保存済みCは再推論せず、Dのseed別・平均・標本標準偏差とCとの差を報告します。')
